In [1]:
print("ok")

ok


In [15]:
import os
from dotenv import load_dotenv
import langchain
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

load_dotenv()
GEMINI_API_KEY=os.getenv("GEMINI_API_KEY")
#this initialize our llm
llm=ChatGoogleGenerativeAI(model="gemini-2.0-flash",
                       temperature=0.7, api_key=GEMINI_API_KEY)

def generate_insights(summary:str)->str:
       prompt = PromptTemplate(
       input_variables=["data_summary"],
       template="""You are a data science expert.

       Analyze the dataset summary below and provide:
       1. Key insights
       2. Data quality issues
       3. Suggestions for cleaning
       4. Potential ML use cases

       Dataset Summary:
       {data_summary}
       """)
       chain=prompt|llm
       response=chain.invoke({"data_summary":summary})
       return response.content

In [16]:
import pandas as pd
from pathlib import Path

def profile_data(file_path: str) -> dict:
 try:
    ext= Path(file_path).suffix
    
    if ext == '.csv':
            df = pd.read_csv(file_path, encoding='utf-8')
    elif ext == '.xlsx':
            df = pd.read_excel(file_path)
    else:
            ValueError("Unsupported file format. Please upload a CSV or Excel file.")
            return None, None, None

    profile = {}

    # Basic info
    profile["shape"] = df.shape
    profile["columns"] = list(df.columns)
    profile["dtypes"] = df.dtypes.astype(str).to_dict()

    # Missing values
    profile["missing_values"] = df.isnull().sum().to_dict()

    # Unique values
    profile["unique_values"] = df.nunique().to_dict()
    # info
    # profile["info"] = df.info()

    # Summary statistics
    profile["summary_stats"] = df.describe(include="all").fillna("").to_dict()

    # Correlation (only numeric)
    numeric_df = df.select_dtypes(include="number")
    if not numeric_df.empty:
        profile["correlation"] = numeric_df.corr().to_dict()
    else:
        profile["correlation"] = {}

    return profile
 except Exception as e:
    print(f"Error profiling data: {e}")
    return None

def profile_to_text(profile: dict) -> str:
    """Convert profile dict into text for LLM"""
    text = []

    text.append(f"Dataset shape: {profile['shape']}")
    text.append(f"Columns: {profile['columns']}")
    text.append(f"Data types: {profile['dtypes']}")

    text.append("Missing values:")
    text.append(str(profile["missing_values"]))

    text.append("Unique values:")
    text.append(str(profile["unique_values"]))

    text.append("Summary stats:")
    text.append(str(profile["summary_stats"]))

    text.append("Correlations:")
    text.append(str(profile["correlation"]))

    return "\n".join(text)

In [18]:
def run_profiling_pipeline(file_path: str):
    # Step 1: Profile dataset
    profile = profile_data(file_path)

    # Step 2: Convert to text for LLM
    summary_text = profile_to_text(profile)
    

    # Step 3: Generate AI insights
    insights = generate_insights(summary_text)

    return {
        "profile": profile,
        "insights": insights
    }
print(run_profiling_pipeline("./insurance.csv"))
# print(type(run_profiling_pipeline("./insurance.csv")))

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 31.964725471s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url